In [1]:
# 1. IMPORTS
import pandas as pd
import numpy as np

# 2. LOAD DATASET FROM NOTEBOOK 02 (includes hour_of_day, day_of_week)
df = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-3-dataset/train_with_time_features.parquet')
print(df.shape)
print(df.columns.tolist()[:20])  # quick sanity check on columns

(590540, 438)
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3']


In [2]:
# 3. ENCODE M-COLUMNS: NaN → EXPLICIT "not_applicable" CATEGORY
m_cols = ['M1','M2','M3','M4','M5','M6','M7','M8','M9']

for col in m_cols:
    print(f"{col} before:", df[col].unique())

for col in m_cols:
    df[col] = df[col].fillna('not_applicable')

print("\nAfter encoding:")
for col in m_cols:
    print(f"{col}:", df[col].value_counts())

M1 before: ['T' None 'F']
M2 before: ['T' None 'F']
M3 before: ['T' None 'F']
M4 before: ['M2' 'M0' None 'M1']
M5 before: ['F' 'T' None]
M6 before: ['T' 'F' None]
M7 before: [None 'F' 'T']
M8 before: [None 'F' 'T']
M9 before: [None 'F' 'T']

After encoding:
M1: M1
T                 319415
not_applicable    271100
F                     25
Name: count, dtype: int64
M2: M2
T                 285468
not_applicable    271100
F                  33972
Name: count, dtype: int64
M3: M3
not_applicable    271100
T                 251731
F                  67709
Name: count, dtype: int64
M4: M4
not_applicable    281444
M0                196405
M2                 59865
M1                 52826
Name: count, dtype: int64
M5: M5
not_applicable    350482
F                 132491
T                 107567
Name: count, dtype: int64
M6: M6
F                 227856
T                 193324
not_applicable    169360
Name: count, dtype: int64
M7: M7
not_applicable    346265
F                 211374
T           

In [3]:
# 4a. INSPECT DeviceInfo VALUES TO FIND NATURAL GROUPING PATTERNS
print(df['DeviceInfo'].value_counts(dropna=False).head(30))

DeviceInfo
None                              471874
Windows                            47722
iOS Device                         19782
MacOS                              12573
Trident/7.0                         7440
rv:11.0                             1901
rv:57.0                              962
SM-J700M Build/MMB29K                549
SM-G610M Build/MMB29K                461
SM-G531H Build/LMY48B                410
rv:59.0                              362
SM-G935F Build/NRD90M                334
SM-G955U Build/NRD90M                328
SM-G532M Build/MMB29T                316
ALE-L23 Build/HuaweiALE-L23          312
SM-G950U Build/NRD90M                290
SM-G930V Build/NRD90M                274
rv:58.0                              269
rv:52.0                              256
SAMSUNG                              235
SM-G950F Build/NRD90M                225
Moto G (4) Build/NPJ25.93-14.7       219
SM-N950U Build/NMF26X                209
SM-A300H Build/LRX22G                203
hi621

In [4]:
# 4b. GROUP DeviceInfo INTO BROADER CATEGORIES BASED ON OBSERVED STRING PATTERNS
def group_device(val):
    if pd.isna(val):
        return 'missing'
    val_str = str(val)
    if 'SM-' in val_str or 'SAMSUNG' in val_str.upper():
        return 'samsung'
    elif 'iOS' in val_str:
        return 'ios'
    elif 'Windows' in val_str:
        return 'windows'
    elif 'MacOS' in val_str:
        return 'macos'
    elif val_str.startswith('rv:') or 'Trident' in val_str:
        return 'browser_only_signature'
    elif any(x in val_str for x in ['Moto', 'ALE-', 'CAM-', 'hi6210', 'HUAWEI']):
        return 'other_android_brand'
    else:
        return 'other'

df['DeviceInfo_grouped'] = df['DeviceInfo'].apply(group_device)
print(df['DeviceInfo_grouped'].value_counts())

DeviceInfo_grouped
missing                   471874
windows                    47775
ios                        19782
macos                      12573
samsung                    11940
browser_only_signature     11825
other                       9414
other_android_brand         5357
Name: count, dtype: int64


In [5]:
# 4c. INSPECT WHAT'S FALLING INTO "other" TO SEE IF WE MISSED A PATTERN
other_devices = df[df['DeviceInfo_grouped'] == 'other']['DeviceInfo'].value_counts()
print(other_devices.head(30))

DeviceInfo
LG-M700 Build/NMF26X                    169
LG-D693n Build/LRX22G                   150
LG-K410 Build/LRX22G                    136
LG-K580 Build/MRA58K                    117
LG-X230 Build/MRA58K                    111
Linux x86_64                            102
F3213 Build/36.1.A.1.86                  98
BLADE V8 SE Build/NRD90M                 97
HTC Desire 10 lifestyle Build/MMB29M     94
XT1635-02 Build/NPN26.118-22-2           94
LG-K500 Build/MMB29M                     92
LG-H870 Build/NRD90U                     86
LG-K200 Build/MXB48T                     85
LG-X180g Build/LMY47I                    84
Blade V6 Plus Build/MRA58K               76
LG-K530 Build/MMB29M                     75
F3113 Build/33.3.A.1.97                  73
LG-D320 Build/KOT49I.V10a                71
HTC Desire 626s Build/LMY47O             67
LG-H542 Build/MRA58K                     67
CHC-U03 Build/HuaweiCHC-U03              67
LG-X240 Build/MRA58K                     65
KFFOWI Build/LVY48F  

In [6]:
# 4d. UPDATE GROUPING TO ADD "lg" AS ITS OWN CATEGORY
def group_device(val):
    if pd.isna(val):
        return 'missing'
    val_str = str(val)
    if 'SM-' in val_str or 'SAMSUNG' in val_str.upper():
        return 'samsung'
    elif 'iOS' in val_str:
        return 'ios'
    elif 'Windows' in val_str:
        return 'windows'
    elif 'MacOS' in val_str:
        return 'macos'
    elif val_str.startswith('rv:') or 'Trident' in val_str:
        return 'browser_only_signature'
    elif val_str.startswith('LG-'):
        return 'lg'
    elif any(x in val_str for x in ['Moto', 'ALE-', 'CAM-', 'hi6210', 'HUAWEI', 'CHC-']):
        return 'other_android_brand'
    else:
        return 'other'

df['DeviceInfo_grouped'] = df['DeviceInfo'].apply(group_device)
print(df['DeviceInfo_grouped'].value_counts())

DeviceInfo_grouped
missing                   471874
windows                    47775
ios                        19782
macos                      12573
samsung                    11940
browser_only_signature     11825
other                       7016
other_android_brand         5424
lg                          2331
Name: count, dtype: int64


In [7]:
# 5a. MERGE RARE card6 CATEGORIES INTO THEIR CLOSEST MATCH
print("Before:", df['card6'].value_counts(dropna=False))

df['card6'] = df['card6'].replace({
    'charge card': 'credit',        # charge card functions like a credit card
    'debit or credit': 'debit'      # ambiguous label, defaulting to more common category
})

print("\nAfter:", df['card6'].value_counts(dropna=False))

Before: card6
debit              439938
credit             148986
None                 1571
debit or credit        30
charge card            15
Name: count, dtype: int64

After: card6
debit     439968
credit    149001
None        1571
Name: count, dtype: int64


In [8]:
# 5b. CONVERT CATEGORICAL COLUMNS TO PANDAS 'category' DTYPE (LightGBM-native handling)
categorical_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
                     'M1','M2','M3','M4','M5','M6','M7','M8','M9', 'DeviceInfo_grouped', 'DeviceType']

for col in categorical_cols:
    df[col] = df[col].astype('category')

print(df[categorical_cols].dtypes)

ProductCD             category
card4                 category
card6                 category
P_emaildomain         category
R_emaildomain         category
M1                    category
M2                    category
M3                    category
M4                    category
M5                    category
M6                    category
M7                    category
M8                    category
M9                    category
DeviceInfo_grouped    category
DeviceType            category
dtype: object


In [9]:
# 6a. COMPUTE FULL CORRELATION MATRIX ACROSS ALL V-COLUMNS
v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
v_corr = df[v_cols].corr().abs()

# 6b. FIND COLUMNS TO DROP: FOR EACH PAIR CORRELATED > 0.95, DROP ONE
upper_triangle = v_corr.where(np.triu(np.ones(v_corr.shape), k=1).astype(bool))
to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.95)]

print(f"Total V-columns: {len(v_cols)}")
print(f"Columns to drop (correlation > 0.95 with another): {len(to_drop)}")
print(f"Columns remaining: {len(v_cols) - len(to_drop)}")

Total V-columns: 339
Columns to drop (correlation > 0.95 with another): 124
Columns remaining: 215


In [10]:
# 6c. CHECK HOW THE DROP COUNT CHANGES AT DIFFERENT THRESHOLDS
for threshold in [0.90, 0.95, 0.99]:
    to_drop_test = [col for col in upper_triangle.columns if any(upper_triangle[col] > threshold)]
    print(f"Threshold {threshold}: {len(to_drop_test)} columns would be dropped, {339 - len(to_drop_test)} remain")

Threshold 0.9: 181 columns would be dropped, 158 remain
Threshold 0.95: 124 columns would be dropped, 215 remain
Threshold 0.99: 36 columns would be dropped, 303 remain


In [11]:
# 6d. DROP THE REDUNDANT V-COLUMNS (threshold = 0.95)
print("Shape before dropping:", df.shape)

df = df.drop(columns=to_drop)

print("Shape after dropping:", df.shape)
print(f"Dropped {len(to_drop)} redundant V-columns")

Shape before dropping: (590540, 439)
Shape after dropping: (590540, 315)
Dropped 124 redundant V-columns


In [12]:
# 7. VERIFY D-COLUMNS AND IDENTITY COLUMNS STILL HAVE THEIR ORIGINAL NaN STRUCTURE (UNTOUCHED)
d_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]
id_cols = [c for c in df.columns if c.startswith('id_')]

print("D-columns missing %:")
print(df[d_cols].isnull().mean() * 100)

print("\nSample id_ columns missing %:")
print(df[id_cols[:5]].isnull().mean() * 100)

D-columns missing %:
D1      0.214888
D2     47.549192
D3     44.514851
D4     28.604667
D5     52.467403
D6     87.606767
D7     93.409930
D8     87.312290
D9     87.312290
D10    12.873302
D11    47.293494
D12    89.041047
D13    89.509263
D14    89.469469
D15    15.090087
dtype: float64

Sample id_ columns missing %:
id_01    75.576083
id_02    76.145223
id_03    88.768923
id_04    88.768923
id_05    76.823755
dtype: float64


In [13]:
# 8a. FINAL LEAKAGE CHECK — MAKE SURE NO COLUMN PERFECTLY PREDICTS isFraud
# (a column with correlation ~1.0 with isFraud would be suspicious — could mean leaked info)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'isFraud']
leak_check = df[numeric_cols + ['isFraud']].corr()['isFraud'].abs().sort_values(ascending=False)
print(leak_check.head(10))

isFraud    1.000000
V257       0.383060
V246       0.366878
V242       0.360590
V201       0.328005
V200       0.318783
V189       0.308219
V188       0.303582
V258       0.297151
V45        0.281832
Name: isFraud, dtype: float64


In [14]:
# 8b. SAVE MODEL-READY DATASET
df.to_parquet('/kaggle/working/train_model_ready.parquet', index=False)
print("Saved:", df.shape)

Saved: (590540, 315)
